# Pure Text-Driven Video Generation Training (EchoMimic Adaptation)

This notebook helps you train a text-to-feature mapping model to drive EchoMimic without speech audio.

## 1. Environment Setup

In [ ]:
!cp drive/MyDrive/playground.zip playground.zip

In [ ]:
# Mount Google Drive if needed
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import sys
import nltk

# Install system dependencies
!apt-get update -qq
!apt-get install -y -qq libavformat-dev libavcodec-dev libavdevice-dev libavutil-dev libswscale-dev libswresample-dev libavfilter-dev pkg-config

# Install python packages with versions compatible with Python 3.12
# Using torch>=2.2.0 as 2.1.2 is not available for 3.12
# Updated mediapipe to 0.10.13 as 0.10.9 is not found
!pip install -q "numpy<2.0.0" "torch>=2.2.0" "torchvision>=0.17.0" "torchaudio>=2.2.0" \
    g2p_en nltk opencv-python matplotlib diffusers==0.24.0 transformers==4.38.1 \
    accelerate omegaconf==2.3.0 einops==0.4.1 mediapipe>=0.10.13 moviepy==1.0.3 \
    facenet-pytorch==2.5.0 gradio

# Force NumPy 1.x again to ensure compatibility with EchoMimic modules
!pip install -q "numpy<2.0.0"

# Download required NLTK data to prevent training crashes
import nltk
nltk.download('averaged_perceptron_tagger_eng', quiet=True)

print("Environment setup complete. PLEASE RESTART THE RUNTIME (Runtime -> Restart session) now to apply NumPy changes.")

In [ ]:
import sys
# Standardize on ffmpeg-python which is the most common wrapper for EchoMimic
!pip uninstall -y python-ffmpeg ffmpeg
!pip install -q ffmpeg-python
print("ffmpeg-python installed and conflicting packages removed.")

In [ ]:
!pip install -q av
print('PyAV (av) installed.')

In [ ]:
import sys
# Ensure huggingface_hub is <= 0.25.0 as per EchoMimic requirements
# Also ensuring transformers is in a range compatible with this hub version
!pip install -q "huggingface_hub<=0.25.0" "transformers>=4.38.1,<4.41.0"
print("Dependencies adjusted. Please restart the runtime if you still see ImportErrors.")

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import re
import cv2
import numpy as np
import nltk
from g2p_en import G2p
import matplotlib.pyplot as plt

# 确保下载 NLTK 依赖
try:
    nltk.data.find('taggers/averaged_perceptron_tagger')
except LookupError:
    nltk.download('averaged_perceptron_tagger', quiet=True)
    nltk.download('averaged_perceptron_tagger_eng', quiet=True)

## 2. Clone Repository & Setup EchoMimic

Assuming you have uploaded the project files to your Drive or you are cloning them.

In [ ]:

!unzip playground.zip -d /content/playground

## 3. Data Preparation

Upload your video dataset (e.g., AVDigits) to `data/asserts/AVDigits`.

In [ ]:
!python /content/playground/src/download_models.py

## 4. Start Training

Run the distillation training script to learn the text-to-feature mapping.

In [ ]:
import ffmpeg
# In ffmpeg-python, the error class is usually accessed via ffmpeg.Error
try:
   # This is just a check to see if the module is loaded correctly
   print(f"FFmpeg module location: {ffmpeg.__file__}")
   # If you need to catch ffmpeg errors, use ffmpeg.Error
   print("FFmpeg Error class is accessible.")
except AttributeError:
   print("FFmpeg Error class still not found. Try restarting the runtime.")

In [ ]:
%cd /content/playground
!python src/preprocess_grid.py

In [ ]:
%cd /content/playground
!python src/train_text_to_feature.py

## 5. Inference (Test the pure text-driven flow)

Once the model is trained (`pretrained_weights/text_driven_model.pth`), you can generate video from pure text.

### 5.1 Download Pretrained Weights
The inference script expects EchoMimic weights and the SD VAE to be present in the `pretrained_weights` folder. Let's download them.

In [ ]:
!rm -rf /content/playground/third_party/EchoMimic/pretrained_weights/sd-image-variations-diffusers

In [ ]:
%cd /content/playground
import sys
import os

# Re-running the generation script after the huggingface_hub fix
!python src/text_driven_prototype.py

# View the generation result
from IPython.display import Video
if os.path.exists("output/text_driven_result/sample_video.mp4"):
    display(Video("output/text_driven_result/sample_video.mp4", embed=True))
else:
    print("Video file not found. Please check the logs above for errors.")